# APP Fraud Mule-Chain Demo (FR-030)

This notebook provides an additive, deterministic demo of APP fraud mule-chain detection.
It uses a mocked rapid-transfer chain and renders standardized investigator outputs.

In [ ]:
from datetime import datetime, timedelta, timezone

import pandas as pd

from notebook_config import (
    init_notebook,
    mask_pii,
    render_decision_block,
    render_reason_codes_block,
)
from banking.analytics.detect_mule_chains import MuleChainDetector

In [ ]:
config = init_notebook(check_env=True, check_services=False)
config

In [ ]:
detector = MuleChainDetector()

base_time = datetime(2026, 1, 15, 12, 0, tzinfo=timezone.utc)
mock_path = {
    "accounts": ["ACC-VICTIM-8F3A", "ACC-MULE1-2B1C", "ACC-MULE2-4D5E", "ACC-EXIT-9123"],
    "transactions": [
        {
            "transaction_id": "TX-APP-001",
            "amount": 5000.00,
            "transaction_date": base_time.isoformat(),
        },
        {
            "transaction_id": "TX-APP-002",
            "amount": 4800.00,
            "transaction_date": (base_time + timedelta(minutes=8)).isoformat(),
        },
        {
            "transaction_id": "TX-APP-003",
            "amount": 4620.00,
            "transaction_date": (base_time + timedelta(minutes=19)).isoformat(),
        },
    ],
}

alert = detector._build_alert_from_path(mock_path, min_hops=3, max_hop_minutes=60)
alert

In [ ]:
if alert:
    evidence_df = pd.DataFrame(
        {
            "metric": [
                "Alert ID",
                "Victim Account",
                "Mule Accounts",
                "Cash-out Account",
                "Hop Count",
                "Total Value",
                "Average Hop Minutes",
                "Risk Score",
            ],
            "value": [
                alert.alert_id,
                mask_pii(alert.victim_account_id),
                ", ".join(mask_pii(m) for m in alert.mule_account_ids),
                mask_pii(alert.cash_out_account_id),
                alert.hop_count,
                f"${alert.total_value:,.2f}",
                round(alert.average_hop_minutes, 2),
                round(alert.risk_score, 3),
            ],
        }
    )
    display(evidence_df)
else:
    print("No mule-chain alert generated in demo scenario.")

In [ ]:
if alert:
    render_decision_block(
        decision="BLOCK & ESCALATE",
        confidence=95,
        action="Freeze chain accounts and escalate to Fraud Operations within SLA.",
        why=(
            f"Detected rapid multi-hop movement from {mask_pii(alert.victim_account_id)} "
            f"to {mask_pii(alert.cash_out_account_id)} via {len(alert.mule_account_ids)} intermediary accounts."
        ),
        evidence=(
            f"HopCount={alert.hop_count}, AvgHopMinutes={alert.average_hop_minutes:.1f}, "
            f"TotalValue=${alert.total_value:,.2f}, RiskScore={alert.risk_score:.3f}"
        ),
    )

    render_reason_codes_block(
        reason_codes=[
            "APP_MULE_CHAIN",
            "RAPID_TRANSFER_VELOCITY",
            "MULTI_HOP_CASH_OUT_PATH",
        ],
        rationale=(
            "Funds traversed three sequential hops within one hour and exhibited "
            "decreasing value transfer consistent with mule-chain fee extraction."
        ),
    )